In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from zen_garden.postprocess.results import Results

# Dataset setup
dataset_name_1 = 'ZEN-Model_HP'
dataset_name_2 = 'ZEN-Model_HP_new'
r1 = Results(os.path.join("outputs", dataset_name_1))
r2 = Results(os.path.join("outputs", dataset_name_2))

# Settings
variables = ['demand']
scenarios_to_include = ['scenario_', 'scenario_S2']
target_name = 'HP'  # Can be in 'carrier', 'technology', or 'node'

dataset_colors = {dataset_name_1: 'tab:blue', dataset_name_2: 'tab:orange'}
scenario_styles = {"scenario_": '-', "scenario_S2": '--'}

for var in variables:
    try:
        # Load and filter by scenario
        df1 = r1.get_full_ts(var).reset_index().rename(columns={"level_0": "scenario"})
        df2 = r2.get_full_ts(var).reset_index().rename(columns={"level_0": "scenario"})

        df1 = df1[df1["scenario"].isin(scenarios_to_include)]
        df2 = df2[df2["scenario"].isin(scenarios_to_include)]

        time_cols = [col for col in df1.columns if isinstance(col, (int, float))]

        # Determine filtering column
        possible_keys = ['carrier', 'technology', 'node']
        group_key = next((k for k in possible_keys if k in df1.columns and k in df2.columns), None)

        if not group_key:
            print(f"⚠️ Skipping {var}: no recognized group key column.")
            continue

        # Filter for HP
        df1 = df1[df1[group_key] == target_name]
        df2 = df2[df2[group_key] == target_name]

        grouped1 = df1.groupby("scenario")[time_cols].sum()
        grouped2 = df2.groupby("scenario")[time_cols].sum()

        # Plot
        plt.figure(figsize=(12, 6))
        for dataset_name, grouped in zip([dataset_name_1, dataset_name_2], [grouped1, grouped2]):
            for scenario in scenarios_to_include:
                if scenario in grouped.index:
                    plt.plot(
                        time_cols,
                        grouped.loc[scenario],
                        label=f"{dataset_name} — {scenario}",
                        color=dataset_colors[dataset_name],
                        linestyle=scenario_styles[scenario]
                    )

        plt.title(f"{var.replace('_', ' ').title()} — {group_key.capitalize()}: {target_name}")
        plt.xlabel("Year")
        plt.ylabel(var)
        plt.grid(True, linestyle="--")
        plt.legend()
        plt.tight_layout()

        xticks_labels = [str(2022 + i) for i in range(len(time_cols))]
        plt.xticks(ticks=time_cols, labels=xticks_labels)

        plt.show()

    except Exception as e:
        print(f"⚠️ Error processing '{var}': {e}")
